# Multiclass Fish Image Classification
This notebook is designed to run in **Google Colab**. 
It covers:
1. Mounting Google Drive & Extracting Data
2. Data Preprocessing & Augmentation
3. Building a Custom CNN
4. Fine-Tuning Pre-trained Models (VGG16, ResNet50, MobileNet, InceptionV3, EfficientNetB0)
5. Model Evaluation
6. Saving the Best Model

In [ ]:
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import VGG16, ResNet50, MobileNet, InceptionV3, EfficientNetB0
from sklearn.metrics import classification_report, confusion_matrix

## 1. Mount Google Drive and Extract Dataset
Ensure your `Dataset.zip` is placed in your Google Drive. Update the path below if necessary.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Update this path to where your Dataset.zip is located in Google Drive
zip_path = '/content/drive/MyDrive/Dataset.zip'
extract_path = '/content/Dataset'

if not os.path.exists(extract_path):
    print("Extracting dataset...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print("Extraction complete.")
else:
    print("Dataset already extracted.")
    
# Check the directory structure
dataset_dir = extract_path # Adjust this if the zip extracts into a subfolder like /content/Dataset/Dataset
print("Classes found:", os.listdir(dataset_dir))

## 2. Data Preprocessing and Augmentation

In [ ]:
batch_size = 32
img_height = 224
img_width = 224

# Data Augmentation for Training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2 # Use 20% of data for validation
)

print("Training Data:")
train_generator = train_datagen.flow_from_directory(
    dataset_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

print("Validation Data:")
validation_generator = train_datagen.flow_from_directory(
    dataset_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

num_classes = train_generator.num_classes
class_names = list(train_generator.class_indices.keys())
print("Classes:", class_names)

## 3. Train Custom CNN from Scratch

In [ ]:
def create_custom_cnn():
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=(img_height, img_width, 3)),
        MaxPooling2D(2, 2),
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D(2, 2),
        Conv2D(128, (3, 3), activation='relu'),
        MaxPooling2D(2, 2),
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

cnn_model = create_custom_cnn()
cnn_model.summary()

# Train the model (Uncomment to run)
# history_cnn = cnn_model.fit(
#     train_generator,
#     validation_data=validation_generator,
#     epochs=10
# )

## 4. Transfer Learning Models
We will create a function to easily instantiate and fine-tune pre-trained models.

In [ ]:
def create_transfer_model(base_model):
    base_model.trainable = False # Freeze base model weights
    
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)
    predictions = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=base_model.input, outputs=predictions)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# Define base models
# models_dict = {
#     'VGG16': VGG16(weights='imagenet', include_top=False, input_shape=(img_height, img_width, 3)),
#     'ResNet50': ResNet50(weights='imagenet', include_top=False, input_shape=(img_height, img_width, 3)),
#     'MobileNet': MobileNet(weights='imagenet', include_top=False, input_shape=(img_height, img_width, 3)),
#     'InceptionV3': InceptionV3(weights='imagenet', include_top=False, input_shape=(img_height, img_width, 3)),
#     'EfficientNetB0': EfficientNetB0(weights='imagenet', include_top=False, input_shape=(img_height, img_width, 3))
# }

# Example Training with VGG16
# vgg_base = VGG16(weights='imagenet', include_top=False, input_shape=(img_height, img_width, 3))
# vgg_model = create_transfer_model(vgg_base)
# history_vgg = vgg_model.fit(train_generator, validation_data=validation_generator, epochs=10)

## 5. Evaluation and Visualization
Plot the training vs validation accuracy and loss.

In [ ]:
def plot_history(history, title):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    # Accuracy
    ax1.plot(history.history['accuracy'], label='Train')
    ax1.plot(history.history['val_accuracy'], label='Validation')
    ax1.set_title(title + ' - Accuracy')
    ax1.legend()
    
    # Loss
    ax2.plot(history.history['loss'], label='Train')
    ax2.plot(history.history['val_loss'], label='Validation')
    ax2.set_title(title + ' - Loss')
    ax2.legend()
    plt.show()

# plot_history(history_cnn, "Custom CNN")
# plot_history(history_vgg, "VGG16")

## 6. Save the Best Model
Once you identify the best performing model, save it to download to your local machine.

In [ ]:
# Example: Saving the VGG16 model if it performed best
# vgg_model.save('best_model.h5')

# You can then download it directly from Colab to your local machine:
from google.colab import files
# files.download('best_model.h5')